#### Домашнее задание

**Датасет:** [`ag_news`](https://huggingface.co/datasets/fancyzhx/ag_news) — классификация новостей по 4-м категориям (World, Sports, Business, Sci/Tech)

**Техническое задание:**

1.  Загрузите датасет `ag_news`
2.  Выберите модель для дообучения (например, `distilbert-base-uncased` или `bert-base-uncased`), `num_labels=4`
3.  Токенизируйте данные (`max_length=128`)
4.  Настройте `TrainingArguments`:
    *   `learning_rate = 2e-5`
    *   `per_device_train_batch_size = 16`
    *   `num_train_epochs = 3`
    *   `eval_strategy = "epoch"`
    *   `save_strategy = "epoch"`
    *   `load_best_model_at_end = True`
    *   `metric_for_best_model = "accuracy"`
5.  Обучите модель с помощью `Trainer`. Для метрик используйте `evaluate.load("accuracy")`
6.  Выведите accuracy на тестовой выборке
7.  Сохраните модель в папку `./ag_news_model`
8.  Протестируйте модель на трех новых новостях (вписать вручную), используя `pipeline`. Выведите предсказанный класс и уверенность модели

In [8]:
!pip install transformers datasets evaluate accelerate scikit-learn

import torch
print(f"GPU доступен: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Тип GPU: {torch.cuda.get_device_name(0)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.4 MB/s eta 0:00:00
GPU доступен: True
Тип GPU: Tesla T4


In [9]:
import numpy as np
import torch
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)
from datasets import load_dataset
import evaluate

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [10]:
dataset = load_dataset("fancyzhx/ag_news")
print(f"Датасет загружен. Train: {len(dataset['train'])}, Test: {len(dataset['test'])}")

README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

Датасет загружен. Train: 120000, Test: 7600


In [11]:
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=4
)
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )
tokenized_datasets = dataset.map(tokenize_function, batched=True)
train_dataset = tokenized_datasets["train"]
test_dataset = tokenized_datasets["test"]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [12]:
metric = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return metric.compute(predictions=predictions, references=labels)

training_args = TrainingArguments(
    output_dir="./ag_news_results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_dir="./logs",
    logging_steps=100,
    save_total_limit=2,
    push_to_hub=False,
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [13]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)
trainer.train()
eval_results = trainer.evaluate()
print(f"Accuracy на тестовой выборке: {eval_results['eval_accuracy']:.4f}")
model.save_pretrained("./ag_news_model")
tokenizer.save_pretrained("./ag_news_model")

Epoch,Training Loss,Validation Loss,Accuracy
1,0.196306,0.177706,0.941316
2,0.118688,0.181627,0.949868
3,0.089136,0.206238,0.946711


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy
0.089136,0.181627,3,0.949868


Accuracy на тестовой выборке: 0.9499


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./ag_news_model/tokenizer_config.json', './ag_news_model/tokenizer.json')

In [14]:
from transformers import pipeline
classifier = pipeline(
    "text-classification",
    model="./ag_news_model",
    tokenizer="./ag_news_model"
)

test_news = [
    "Apple Inc. announced a new iPhone with advanced AI features and improved camera system.",
    "Lionel Messi steps off bench and scores to cap Argentina’s World Cup win over Jordan.",
    "Scientists discovered a new species of deep-sea fish in the Pacific Ocean.",
]

id_to_label = {
    0: "World",
    1: "Sports",
    2: "Business",
    3: "Sci/Tech"
}

for i, news in enumerate(test_news, 1):
    result = classifier(news)
    label_id = int(result[0]['label'].split('_')[-1])
    confidence = result[0]['score']

    print(f"\nНовость {i}:")
    print(f"Текст: {news[:100]}...")
    print(f"Предсказанный класс: {id_to_label.get(label_id, f'Класс {label_id}')}")
    print(f"Уверенность модели: {confidence:.4f}")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


Новость 1:
Текст: Apple Inc. announced a new iPhone with advanced AI features and improved camera system....
Предсказанный класс: Sci/Tech
Уверенность модели: 0.9916

Новость 2:
Текст: Lionel Messi steps off bench and scores to cap Argentina’s World Cup win over Jordan....
Предсказанный класс: World
Уверенность модели: 0.9968

Новость 3:
Текст: Scientists discovered a new species of deep-sea fish in the Pacific Ocean....
Предсказанный класс: Sci/Tech
Уверенность модели: 0.9742
